# Outcome-constrained Bayesian Optimization

未知のblack-box制約をsurrogate posteriorで学習し、objective improvementとfeasibilityを同じMC acquisitionで合成します。

Theory: `docs/theory/acquisition/08_constraints.md`  
Runtime: `docs/optimization/acquisition-integration.md`


In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.objective import GenericMCObjective
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import ModelListGP, SingleTaskGP
from robotorchan.optim import optimize_acqf

torch.manual_seed(0)
dtype = torch.double


## Problem setup

objectiveは最大化します。未知制約は `c(x) <= 0` をfeasibleとします。ここでは `0.2 <= x <= 0.8` が真のfeasible領域ですが、candidate生成時にはその境界を既知情報として使いません。


In [ ]:
def evaluate(X):
    objective = 1.0 - (X - 0.72).square()
    upper_constraint = X - 0.8
    lower_constraint = 0.2 - X
    return objective, upper_constraint, lower_constraint

train_X = torch.linspace(0.0, 1.0, 9, dtype=dtype).unsqueeze(-1)
objective_Y, upper_Y, lower_Y = evaluate(train_X)


## Model objective and constraint outputs

objectiveと2つのconstraintを別GPとして学習し、`ModelListGP`で同じposterior sampleへまとめます。


In [ ]:
model = ModelListGP(
    SingleTaskGP(train_X, objective_Y),
    SingleTaskGP(train_X, upper_Y),
    SingleTaskGP(train_X, lower_Y),
)
mll = model.make_mll()
model.fit(mll)

test_X = torch.linspace(0.0, 1.0, 101, dtype=dtype).unsqueeze(-1)
posterior = model.posterior(test_X)
assert posterior.mean.shape[-1] == 3


## Objective and outcome constraints

`GenericMCObjective`はraw model samplesのoutput 0だけをobjective utilityにします。constraint callableはraw samplesを受け、BoTorchのMC constrained acquisition conventionに従って非正値をfeasibleとします。


In [ ]:
objective = GenericMCObjective(lambda Y, X=None: Y[..., 0])
constraints = [
    lambda Y: Y[..., 1],
    lambda Y: Y[..., 2],
]
sampler = SobolQMCNormalSampler(torch.Size([64]), seed=27)

feasible_observed = (upper_Y <= 0) & (lower_Y <= 0)
best_f = objective_Y[feasible_observed].max()
acqf = qLogExpectedImprovement(
    model=model,
    best_f=best_f,
    sampler=sampler,
    objective=objective,
    constraints=constraints,
)


## Candidate generation

探索boundsは `[0, 1]` のままです。`0.2 <= x <= 0.8` をoptimizer constraintとして渡していない点が重要です。feasibilityはconstraint posteriorを通じてacquisition内で確率的に扱われます。


In [ ]:
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
candidate, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
candidate_objective, candidate_upper, candidate_lower = evaluate(candidate)
print("candidate:", candidate)
print("objective:", candidate_objective)
print("constraints:", candidate_upper, candidate_lower)
assert candidate.shape == torch.Size([1, 1])
assert torch.isfinite(acq_value).all()


## Update once

実験後はobjectiveだけでなくconstraint outputsも観測し、すべてのsurrogate training dataへ追加します。


In [ ]:
updated_X = torch.cat([train_X, candidate.detach()], dim=0)
updated_objective = torch.cat([objective_Y, candidate_objective.detach()], dim=0)
updated_upper = torch.cat([upper_Y, candidate_upper.detach()], dim=0)
updated_lower = torch.cat([lower_Y, candidate_lower.detach()], dim=0)
assert updated_X.shape[0] == train_X.shape[0] + 1
assert updated_objective.shape == updated_upper.shape == updated_lower.shape


## Responsibility boundary

- **Outcome / black-box constraint**: 評価するまで未知。surrogate + constrained acquisitionで扱います。
- **Candidate / input-space constraint**: Xから既知。`CandidateConstraints`やBoTorch optimizer constraintsで探索領域を制限します。
- 両方を同時に使うこともできますが、一方を他方へ変換しません。
- constrained BOはsafe BOではありません。通常のconstrained acquisitionは制約違反を絶対に防ぐ保証を与えません。
- feasible observationが1点もない場合は、単純なfeasible incumbentベースのEIとは別の初期探索設計が必要です。
